# 5. Full End-to-End Multi-Agent Research System

This notebook integrates the complete three-phase pipeline: Scoping, Multi-Agent Parallel Research, and Final Report Synthesis.

In [ ]:
import os
import asyncio
from typing import List, Dict, Any, Optional
from datetime import datetime
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

load_dotenv()

## 1. System Schemas & State

In [ ]:
class ResearchQuestion(BaseModel):
    title: str
    core_question: str
    sub_topics: List[str]
    target_depth: str = "in-depth"
    search_queries: List[str]

class ResearchNote(BaseModel):
    topic: str
    summary: str
    key_findings: List[str] = Field(default_factory=list)
    sources: List[Dict[str, str]] = Field(default_factory=list)

class ResearchReport(BaseModel):
    title: str
    executive_summary: str
    full_markdown: str
    sources: List[Dict[str, str]]

class MasterState(BaseModel):
    query: str
    brief: Optional[ResearchQuestion] = None
    notes: List[ResearchNote] = Field(default_factory=list)
    report: Optional[ResearchReport] = None

## 2. Pipeline Nodes: Scope, Parallel Research, and Write

In [ ]:
def get_llm():
    return ChatOpenAI(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        openai_api_key=os.getenv("OPENAI_API_KEY"),
        openai_api_base=os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1"),
        temperature=0.2,
    )

async def scope_phase(state: MasterState) -> Dict[str, Any]:
    llm = get_llm().with_structured_output(ResearchQuestion)
    brief = await llm.ainvoke([
        SystemMessage(content="Generate a structured research brief with 3 distinct sub-topics for parallel exploration."),
        HumanMessage(content=f"Topic: {state.query}")
    ])
    return {"brief": brief}

async def research_single_topic(topic: str) -> ResearchNote:
    llm = get_llm()
    resp = await llm.ainvoke([
        SystemMessage(content="You are an expert sub-topic researcher. Analyze the topic with 3 key findings."),
        HumanMessage(content=f"Research sub-topic: {topic}")
    ])
    return ResearchNote(
        topic=topic,
        summary=resp.content,
        key_findings=[l.strip("- ") for l in resp.content.split("\n") if l.strip().startswith("-")],
        sources=[{"title": f"{topic} Reference", "url": "https://example.com"}]
    )

async def research_phase(state: MasterState) -> Dict[str, Any]:
    sub_topics = state.brief.sub_topics if state.brief else ["General Overview"]
    notes = await asyncio.gather(*[research_single_topic(t) for t in sub_topics])
    return {"notes": list(notes)}

async def write_phase(state: MasterState) -> Dict[str, Any]:
    llm = get_llm()
    notes_text = "\n\n".join([f"### {n.topic}\n{n.summary}" for n in state.notes])
    prompt = f"Write a comprehensive Markdown research report on '{state.brief.title}' combining these findings:\n\n{notes_text}"
    
    resp = await llm.ainvoke([
        SystemMessage(content="You are a Principal Technical Writer. Synthesize findings into an exhaustive Markdown report."),
        HumanMessage(content=prompt)
    ])
    
    report = ResearchReport(
        title=state.brief.title,
        executive_summary=f"Synthesized report on {state.brief.title}",
        full_markdown=resp.content,
        sources=[s for n in state.notes for s in n.sources]
    )
    return {"report": report}

## 3. Build & Execute Full Research System

In [ ]:
builder = StateGraph(MasterState)
builder.add_node("scope", scope_phase)
builder.add_node("research", research_phase)
builder.add_node("write", write_phase)

builder.add_edge(START, "scope")
builder.add_edge("scope", "research")
builder.add_edge("research", "write")
builder.add_edge("write", END)

full_system = builder.compile()

# Run End-to-End System
input_query = MasterState(query="Explain Agentic AI Workflows: State Machines, Human-in-the-Loop, and Model Context Protocol")
final_state = await full_system.ainvoke(input_query)

print("\n=================== FINAL SYNTHESIZED REPORT ===================\n")
print(final_state["report"].full_markdown[:1000] + "\n...\n[Report continues]")